# Шунав — natural Tajik voice for every chapter

Reads every chapter in the voice from a short recording, on Colab's free GPU. Two engines (choose in step 1):

- **`mms`** — Meta's open Tajik voice ([facebook/mms-tts-tgk](https://huggingface.co/facebook/mms-tts-tgk),
  CC BY-NC 4.0, no login), then **FreeVC** turns that voice into the one in `voice.wav`. Works right away.
- **`chatterbox`** — **Chatterbox Tajik** ([Tohirju/chatterbox-mtl-tajik](https://huggingface.co/Tohirju/chatterbox-mtl-tajik), MIT),
  copies the voice itself and sounds more natural, but its authors must first approve your access request.

**Before you start (once):**
1. `Runtime → Change runtime type → T4 GPU`.
2. Only for `chatterbox`: on Hugging Face accept the model's conditions (and wait for approval), then make a *read*
   token at https://huggingface.co/settings/tokens — you paste it into the login box in step 3.
3. In Google Drive make a folder **`shunav_tts`** and put into it:
   - `chapters.json` — made on your computer with `.venv\Scripts\python export_for_tts.py`
   - `voice.wav` — 10–20 seconds of one Tajik speaker reading calmly, no music or noise.
     **Only use the voice of someone who agreed to it.**

Then `Runtime → Run all`. Finished chapters are saved to `shunav_tts/audio/<chapter id>.mp3` one by one.
If Colab stops (free sessions end after a few hours), just run all again: finished chapters are skipped.
At the end download the `audio` folder and on your computer run `.venv\Scripts\python import_audio.py <folder>`.

In [ ]:
# 1. Choose the engine, install (about 2 minutes) and check the GPU
ENGINE = 'mms'  # 'mms' works now; 'chatterbox' once Hugging Face approves your access

if ENGINE == 'chatterbox':
    !pip install -q chatterbox-tts lameenc
else:
    !pip install -q coqui-tts lameenc soundfile
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Google Drive: where chapters.json and voice.wav are, and where the audio goes
from google.colab import drive
drive.mount('/content/drive')

import json, os
WORK = '/content/drive/MyDrive/shunav_tts'
OUT = f'{WORK}/audio'
os.makedirs(OUT, exist_ok=True)
VOICE = f'{WORK}/voice.wav'
chapters = json.load(open(f'{WORK}/chapters.json', encoding='utf-8'))
assert os.path.exists(VOICE), 'Put voice.wav into the shunav_tts folder'
done = {name[:-4] for name in os.listdir(OUT) if name.endswith('.mp3')}
print(f'{len(chapters)} chapters, {len(done)} already done')

In [ ]:
# 3. Only for chatterbox: log in to Hugging Face (paste your read token) and download the Tajik model
if ENGINE == 'chatterbox':
    from huggingface_hub import login, snapshot_download
    login()
    MODEL_DIR = snapshot_download('Tohirju/chatterbox-mtl-tajik')
else:
    print('mms needs no login')

In [ ]:
# 4. Load the model(s) and the voice from voice.wav
import torch, tempfile
import numpy as np
torch.manual_seed(7)

if ENGINE == 'chatterbox':
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS
    model = ChatterboxMultilingualTTS.from_local(MODEL_DIR, device='cuda')
    try:
        model.t3.tfmr.config._attn_implementation = 'eager'  # needed with newer transformers (see the model card)
    except AttributeError:
        pass
    RATE = model.sr

    # how it reads: calm and clear (more exaggeration = more emotion, lower cfg = slower)
    SETTINGS = dict(exaggeration=0.45, cfg_weight=0.4, temperature=0.7)
    try:
        model.prepare_conditionals(VOICE, exaggeration=SETTINGS['exaggeration'])  # the voice once, not for every line
        VOICE_ARGS = {}
    except (AttributeError, TypeError):
        VOICE_ARGS = {'audio_prompt_path': VOICE}
else:
    import soundfile
    from transformers import VitsModel, AutoTokenizer
    from TTS.api import TTS

    mms = VitsModel.from_pretrained('facebook/mms-tts-tgk').to('cuda')
    mms_tokens = AutoTokenizer.from_pretrained('facebook/mms-tts-tgk')
    mms.speaking_rate = 0.75        # slower than normal: calm reading (1.0 = normal)
    mms.noise_scale = 0.33          # little randomness: the same steady voice in every sentence
    mms.noise_scale_duration = 0.4  # steady rhythm
    MMS_LETTERS = set(mms_tokens.get_vocab()) - {'_'}

    CONVERT = False  # True = change Meta's voice into the one in voice.wav (FreeVC)
    if CONVERT:
        converter = TTS('voice_conversion_models/multilingual/vctk/freevc24').to('cuda')
        RATE = converter.voice_converter.output_sample_rate
    else:
        RATE = mms.config.sampling_rate

In [ ]:
# 5. Text -> pieces to read (with the pause after each)
import re

ONES = ['', 'як', 'ду', 'се', 'чор', 'панҷ', 'шаш', 'ҳафт', 'ҳашт', 'нӯҳ']
TEENS = ['даҳ', 'ёздаҳ', 'дувоздаҳ', 'сенздаҳ', 'чордаҳ', 'понздаҳ', 'шонздаҳ', 'ҳабдаҳ', 'ҳаждаҳ', 'нуздаҳ']
TENS = ['', '', 'бист', 'сӣ', 'чил', 'панҷоҳ', 'шаст', 'ҳафтод', 'ҳаштод', 'навад']
VOWELS = set('аеёиоуэюяӣӯ')

def join_with_u(words):
    """Tajik joins the parts of a number with -у / -ю: бисту ду, сию се."""
    out = []
    for i, word in enumerate(words):
        if i < len(words) - 1:
            word += 'ю' if word[-1] in VOWELS else 'у'
            if word.endswith('ӣю'):
                word = word[:-2] + 'ию'
        out.append(word)
    return ' '.join(out)

def number_to_words(n):
    if n == 0:
        return 'сифр'
    words = []
    if n >= 1000:
        thousands = n // 1000
        words.append('ҳазор' if thousands == 1 else number_to_words(thousands) + ' ҳазор')
        n %= 1000
    if n >= 100:
        words.append(ONES[n // 100] + 'сад')
        n %= 100
    if n >= 20:
        words.append(TENS[n // 10])
        n %= 10
    elif n >= 10:
        words.append(TEENS[n - 10])
        n = 0
    if n:
        words.append(ONES[n])
    return join_with_u(words)

def clean(text):
    """Keep the punctuation (the voice uses it for melody), fix typos, numbers in words."""
    text = text.replace('ѐ', 'ё').replace('\ufffd', '').replace('◆', '').replace('«', '').replace('»', '')
    text = re.sub(r'(\d+)\s*[–-]\s*(\d+)', r'\1 то \2', text)  # 1–25 = як то бисту панҷ
    text = re.sub(r'\d+', lambda m: number_to_words(int(m.group())), text)
    text = re.sub(r'\s+', ' ', text).strip(' -–—')
    if text and text[-1] not in '.!?…,;:':
        text += '.'
    return text

MAX = 260  # longer pieces make this kind of model mumble

def split_long(text):
    """Prose: by sentences, then by commas, so no piece is longer than MAX."""
    if len(text) <= MAX:
        return [text]
    parts, current = [], ''
    for sentence in re.split(r'(?<=[.!?…;:])\s+|(?<=,)\s+', text):
        if current and len(current) + len(sentence) > MAX:
            parts.append(current)
            current = sentence
        else:
            current = f'{current} {sentence}'.strip()
    return parts + ([current] if current else [])

def pieces(chapter):
    out = [(clean(chapter['title']), 1.2)]
    for stanza in re.split(r'\n\s*\n', chapter['text'].strip()):
        lines = [line.strip() for line in stanza.split('\n') if line.strip()]
        if not lines:
            continue
        if lines[0].startswith('◆'):  # a new poem: its heading, a longer pause before and after
            out[-1] = (out[-1][0], 1.4)
            out.append((clean(lines.pop(0)), 0.9))
        verse = lines and all(len(line) < 70 for line in lines)
        if verse:
            # a bayt (two half-lines) in one breath, like a reader does
            for i in range(0, len(lines), 2):
                bayt = clean(', '.join(line.rstrip('.,;:') for line in lines[i:i + 2]))
                out.append((bayt, 0.55))
        else:
            for text in split_long(clean(' '.join(lines))):
                out.append((text, 0.35))
        if out:
            out[-1] = (out[-1][0], max(out[-1][1], 0.9))
    return [(text, pause) for text, pause in out if text.strip('.,;: ')]

print(pieces(chapters[0])[:6])

In [ ]:
# 6. Reading
import lameenc
from IPython.display import Audio, display

VOICE_SEED = 7  # the same seed before every sentence = the same voice in every sentence

def say(text):
    torch.manual_seed(VOICE_SEED)
    if ENGINE == 'chatterbox':
        with torch.inference_mode():
            wav = model.generate(text, language_id='ru', **VOICE_ARGS, **SETTINGS)  # Tajik is read with the Cyrillic (ru) tokens
        return wav.squeeze().float().cpu().numpy()
    # mms knows only the letters (no punctuation): the pauses come from splitting the text into pieces
    letters = ''.join(c if c in MMS_LETTERS else ' ' for c in text.lower().replace('-', ' '))
    letters = re.sub(r'\s+', ' ', letters).strip()
    if not letters:
        return np.zeros(0, dtype=np.float32)
    with torch.inference_mode():
        wav = mms(**mms_tokens(letters, return_tensors='pt').to('cuda')).waveform.squeeze().float().cpu().numpy()
    if not CONVERT:
        return wav
    with tempfile.TemporaryDirectory() as tmp:
        soundfile.write(f'{tmp}/in.wav', wav, mms.config.sampling_rate)
        converter.voice_conversion_to_file(source_wav=f'{tmp}/in.wav', target_wav=VOICE, file_path=f'{tmp}/out.wav')
        out, _ = soundfile.read(f'{tmp}/out.wav', dtype='float32')
    return out

def read(chapter_pieces):
    sound = []
    for text, pause in chapter_pieces:
        wav = say(text)
        loudness = np.sqrt(np.mean(wav ** 2)) if len(wav) else 0
        if loudness > 1e-4:
            wav = wav * (0.08 / loudness)  # every sentence equally loud
        sound.append(wav)
        sound.append(np.zeros(int(pause * RATE), dtype=np.float32))
    audio = np.concatenate(sound)
    return np.clip(audio / max(np.abs(audio).max(), 1e-6) * 0.9, -1, 1)

def save_mp3(audio, path):
    encoder = lameenc.Encoder()
    encoder.set_bit_rate(64)
    encoder.set_in_sample_rate(RATE)
    encoder.set_channels(1)
    encoder.set_quality(2)
    data = encoder.encode((audio * 32767).astype(np.int16).tobytes()) + encoder.flush()
    with open(path + '.part', 'wb') as out:
        out.write(data)
    os.replace(path + '.part', path)  # a stopped session never leaves half a file

# listen to a short test before reading everything
test = read(pieces(chapters[0])[:5])
display(Audio(test, rate=RATE))

Listen to the test above.
- `mms`: too fast → lower `mms.speaking_rate` in step 4 (e.g. 0.7). `CONVERT = True` changes the voice into
  the one in `voice.wav` (run step 4 and 6 again).
- The voice changes between sentences → try another `VOICE_SEED` in step 6 (any number) and keep the one you like.
- `chatterbox`: too fast → lower `cfg_weight` in step 4 (e.g. 0.3); flat → raise `exaggeration` (e.g. 0.6).

Then run step 7. It saves each chapter as soon as it is ready.

In [ ]:
# 7. All chapters (finished ones are skipped). One chapter of poetry takes a few minutes on a T4.
import time
for number, chapter in enumerate(chapters, 1):
    path = f"{OUT}/{chapter['id']}.mp3"
    if os.path.exists(path):
        continue
    started = time.time()
    audio = read(pieces(chapter))
    save_mp3(audio, path)
    print(f"[{number}/{len(chapters)}] {chapter['book']} — {chapter['title']}: "
          f"{len(audio) / RATE / 60:.0f} min audio in {(time.time() - started) / 60:.0f} min")
print('All chapters are done. Download the shunav_tts/audio folder.')